# Create Neo4j graph for the BART system

University of California, Berkeley

Master of Information and Data Science (MIDS) program

w205 - Fundamentals of Data Engineering



# Included Modules and Packages

Code cell containing your includes for modules and packages

Some starter code is provided

You may change the starter code as needed

You may add as much code and/or as many code cells as you need

In [ ]:
import neo4j

import csv

import math
import numpy as np
import pandas as pd

import psycopg2

# Supporting code

Code cells containing any supporting code, such as connecting to the database, any functions, etc.  

Remember you can freely use any code from the labs. You do not need to cite code from the labs.

Some starter code is provided

You may change the starter code as needed

You may add as much code and/or as many code cells as you need

In [ ]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [ ]:
session = driver.session(database="neo4j")

In [ ]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [ ]:
def my_neo4j_number_nodes_relationships():
    "print the number of nodes and relationships"
   
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    print("-------------------------")
    print("  Nodes:", number_nodes)
    print("  Relationships:", number_relationships)
    print("-------------------------")


In [ ]:
def my_neo4j_create_node(station_name):
    "create a node with label Station"
    
    query = """
    
    CREATE (:Station {name: $station_name})
    
    """
    
    session.run(query, station_name=station_name)
    

In [ ]:
def my_neo4j_create_relationship_one_way(from_station, to_station, weight):
    "create a relationship one way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [ ]:
def my_neo4j_create_relationship_two_way(from_station, to_station, weight):
    "create relationships two way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

# 3.3.1 Wipe out the Neo4j database

Call the function my_neo4j_wipe_out_database() to wipe out the Neo4j database

In [ ]:
my_neo4j_wipe_out_database()

# 3.3.2 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships()  

The output should look similar to this:
```
-------------------------
  Nodes: 0
  Relationships: 0
-------------------------
```

In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 114
  Relationships: 346
-------------------------


# 3.3.5 Query the list of stations and the lines they serve, create line nodes, and create relationships between the line nodes and the departure and arrival nodes with weight 0



In [ ]:
connection.rollback()

query = """

select station,line
from lines
order by station,line

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    line = row[1]
    
    my_neo4j_create_node(line+' '+station)
    

# 3.3.6 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships() 



In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 114
  Relationships: 0
-------------------------


# 3.3.7 Query the list of travel times for each station and create relationship based on travel time


In [ ]:
connection.rollback()

# query = """

# select a.station, a.line as from_line, b.line as to_line, s.transfer_time
# from lines a
#      join lines b
#        on a.station = b.station and a.line <> b.line 
#      join stations s
#        on a.station = s.station
# order by 1, 2, 3

# """

query = """

select line,station_1,station_2,travel_time from travel_times join lines on station=station_1

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    line = row[0]
    from_stn = row[1]
    to_stn = row[2]
    travel_time = int(row[3])
    #transfer_time = int(row[3])
 
    my_neo4j_create_relationship_one_way(line+' '+from_stn,line+' '+to_stn,travel_time)
    my_neo4j_create_relationship_one_way(line+' '+to_stn,line+' '+from_stn,travel_time)

    

# 3.3.8 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships() 

The output should look similar to this:
```
-------------------------
  Nodes: 214
  Relationships: 436
-------------------------
```


In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 114
  Relationships: 218
-------------------------


# 3.3.9 Query the list of all segments between each station and its adjoining stations, create a relationship for each segment both ways




In [ ]:
connection.rollback()

query = """

select line, station from lines;

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()
rows
d = {key[1]:[] for key in rows}
for row in rows:
    d[row[1]].append(row[0])
for stn in d.keys():
    for l in range(0,len(d[stn])-1):
        print(d[stn],stn)
        my_neo4j_create_relationship_one_way(d[stn][l]+' '+stn,d[stn][l+1]+' '+stn,0)
        my_neo4j_create_relationship_one_way(d[stn][l+1]+' '+stn,d[stn][l]+' '+stn,0)

['blue', 'green', 'orange'] Bay Fair
['blue', 'green', 'orange'] Bay Fair
['blue', 'green', 'orange'] San Leandro
['blue', 'green', 'orange'] San Leandro
['blue', 'gray', 'green', 'orange'] Coliseum
['blue', 'gray', 'green', 'orange'] Coliseum
['blue', 'gray', 'green', 'orange'] Coliseum
['blue', 'green', 'orange'] Fruitvale
['blue', 'green', 'orange'] Fruitvale
['blue', 'green', 'orange'] Lake Merritt
['blue', 'green', 'orange'] Lake Merritt
['blue', 'green', 'red', 'yellow'] West Oakland
['blue', 'green', 'red', 'yellow'] West Oakland
['blue', 'green', 'red', 'yellow'] West Oakland
['blue', 'green', 'red', 'yellow'] Embarcadero
['blue', 'green', 'red', 'yellow'] Embarcadero
['blue', 'green', 'red', 'yellow'] Embarcadero
['blue', 'green', 'red', 'yellow'] Montgomery Street
['blue', 'green', 'red', 'yellow'] Montgomery Street
['blue', 'green', 'red', 'yellow'] Montgomery Street
['blue', 'green', 'red', 'yellow'] Powell Street
['blue', 'green', 'red', 'yellow'] Powell Street
['blue', 'g

# 3.3.10 Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships()

The output should look similar to this:

```
-------------------------
  Nodes: 214
  Relationships: 652
-------------------------
```


In [ ]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 114
  Relationships: 252
-------------------------


In [ ]:
query = """

CALL gds.graph.project('bart_map', 'Station', 'LINK', {relationshipProperties: 'weight'})

"""

# CALL gds.betweenness.stream()
# YIELD nodeId, score
# RETURN gds.util.asNode(nodeId).name AS name, score as betweenness
# ORDER BY betweenness DESC

my_neo4j_run_query_pandas(query)

,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'Station': {'label': 'Station', 'properties':...","{'LINK': {'orientation': 'NATURAL', 'indexInve...",bart_map,114,346,1789


Betweeness Centrality

In [ ]:
query = """

CALL gds.betweenness.stream('bart_map')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as betweenness
ORDER BY betweenness DESC

"""



my_neo4j_run_query_pandas(query)

,name,betweenness
0,green West Oakland,3142.520859
1,green Lake Merritt,3066.190791
2,green Fruitvale,2771.667481
3,red West Oakland,2726.483864
4,red 12th Street,2658.950211
...,...,...
109,yellow SFO,6.055576
110,yellow Antioch,0.000000
111,blue Dublin,0.000000
112,red Millbrae,0.000000
